# Using OpenDP Polars: Features

This notebook showcases how researcher could use the Secure Data Disclosure system. It explains the different functionnalities provided by the `lomas_client` library to interact with the secure server.

The secure data are never visible by researchers. They can only access to differentially private responses via queries to the server.

Each user has access to one or multiple projects and for each dataset has a limited budget with $\epsilon$ and $\delta$ values.

In [ ]:
from rich.jupyter import print

%load_ext rich

In [ ]:
# from IPython.display import Image
# Image(filename="images/image_demo_client.png", width=800)

We will use a synthetic dataset about COVID to demonstrate the how to use the library `lomas_client` with polars queries.

## Step 1: Install the library

It can be installed via the pip command:

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
# !pip install lomas_client

In [ ]:
from lomas_client import Client
import numpy as np
import opendp.prelude as dp

## Step 2: Initialise the client

Once the library is installed, a Client object must be created. It is responsible for sending sending requests to the server and processing responses in the local environment. It enables a seamless interaction with the server. 

The client needs a few parameters to be created. Usually, these would be set in the environment by the system administrator and be transparent to lomas users. In this instance, the following code snippet sets a few of these parameters that are specific to this notebook. 

In [ ]:
# The following would usually be set in the environment by a system administrator
# and be tranparent to lomas users.
# Uncomment them if you are running against a Kubernetes deployment.
# They have already been set for you if you are running locally within a devenv or the Jupyter lab set up by Docker compose.

import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_OIDC_DISCOVERY_URL"] = "https://dex.example.com:443/.well-known/openid-configuration"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

os.environ["LOMAS_CLIENT_USER_NAME"] = "mr.corona@example.com"
os.environ["LOMAS_CLIENT_USER_PASSWORD"] = "mr.corona"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "COVID_SYNTHETIC"

# Note that all client settings can also be passed as keyword arguments to the Client constructor.
# eg. client = Client(user_name = "Mr.corona") takes precedence over setting the "LOMAS_CLIENT_USER_NAME"
# environment variable.

In [ ]:
client = Client()

[14:47:47] WARNING  OIDC IdP or Lomas service configured without TLS -> using insecure transport  ]8;id=769835;file:///home/pauline/Desktop/lomas/client/lomas_client/http_client.py\http_client.py]8;;\:]8;id=442526;file:///home/pauline/Desktop/lomas/client/lomas_client/http_client.py#35\35]8;;\

## Step 3: Metadata and dummy dataset

### Getting dataset metadata

The user has never seen the data and as a first step to understand what is available to her, she would like to check the metadata of the dataset. Therefore, she just needs to call the `get_dataset_metadata()` function of the client. As this is public information, this does not cost any budget.

This function returns metadata information in a format based on [SmartnoiseSQL dictionary format](https://docs.smartnoise.org/sql/metadata.html#dictionary-format), where among other, there is information about all the available columns, their type, bound values (see Smartnoise page for more details). Any metadata is required for Smartnoise-SQL is also required here and additional information such that the different categories in a string type column column can be added.

In [ ]:
covid_metadata = client.get_dataset_metadata()
covid_metadata


{
    '@context': ['http://www.w3.org/ns/csvw', '/home/onyxia/work/csvw-eo/csvw-eo-context.jsonld'],
    '@type': 'Table',
    'privacyUnit': 'id',
    'maxContributions': 52,
    'maxLength': 50048,
    'publicLength': 50048,
    'tableSchema': {
        'columns': [
            {
                '@type': 'Column',
                'name': 'id',
                'datatype': <DataTypes.POSITIVE_INTEGER: 'positiveInteger'>,
                'required': True,
                'privacyId': True,
                'nullableProportion': 0.0,
                'minimum': 1,
                'maximum': 2000
            },
            {
                '@type': 'Column',
                'name': 'date',
                'datatype': <DataTypes.DATE: 'date'>,
                'required': True,
                'privacyId': False,
                'nullableProportion': 0.0,
                'minimum': '2022-08-01',
                'maximum': '2023-07-30'
            },
            {
                '@type': 'C

### Get a dummy dataset

Now, that she has seen and understood the metadata, she wants to get an even better understanding of the dataset (but is still not able to see it). A solution to have an idea of what the dataset looks like it to create a dummy dataset. 

Based on the public metadata of the dataset, a random dataframe can be created created. By default, there will be 100 rows and the seed is set to 42 to ensure reproducibility, but these 2 variables can be changed to obtain different dummy datasets.
Getting a dummy dataset does not affect the budget as there is no differential privacy here. It is not a synthetic dataset and all that could be learn here is already present in the public metadata (it is created randomly on the fly based on the metadata).

Dr. FSO first create a dummy dataset with 200 rows and chooses a seed of 0.

In [ ]:
NB_ROWS = 200
SEED = 0

In [ ]:
dummy_lf = client.get_dummy_dataset(nb_rows=NB_ROWS, seed=SEED, lazy=True)
print(dummy_lf.collect())

shape: (200, 13)
┌──────┬────────────────┬──────────┬───────────┬───┬─────────┬────────────────┬───────┬────────────┐
│ id   ┆ date           ┆ temporal ┆ georegion ┆ … ┆ subType ┆ hospitalizatio ┆ death ┆ patient_id │
│ ---  ┆ ---            ┆ ---      ┆ ---       ┆   ┆ ---     ┆ n              ┆ ---   ┆ ---        │
│ i64  ┆ datetime[ns]   ┆ i64      ┆ str       ┆   ┆ str     ┆ ---            ┆ bool  ┆ i64        │
│      ┆                ┆          ┆           ┆   ┆         ┆ bool           ┆       ┆            │
╞══════╪════════════════╪══════════╪═══════════╪═══╪═════════╪════════════════╪═══════╪════════════╡
│ 1120 ┆ 2023-05-24     ┆ 29       ┆ GE        ┆ … ┆ null    ┆ true           ┆ true  ┆ 18708      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 564  ┆ 2023-05-26     ┆ 8        ┆ AG        ┆ … ┆ null    ┆ true           ┆ false ┆ 924        │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 118  ┆ 2023-03-26     ┆ 41       ┆ TG        ┆ … ┆ null    ┆ false          ┆ true  ┆ 4549       │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 755  ┆ 2023-06-10     ┆ 29       ┆ SO        ┆ … ┆ BA.2.75 ┆ true           ┆ false ┆ 28946      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 230  ┆ 2023-02-21     ┆ 30       ┆ VD        ┆ … ┆ null    ┆ true           ┆ false ┆ 19625      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ …    ┆ …              ┆ …        ┆ …         ┆ … ┆ …       ┆ …              ┆ …     ┆ …          │
│ 1443 ┆ 2023-02-21     ┆ 42       ┆ AG        ┆ … ┆ BA.4    ┆ false          ┆ false ┆ 16895      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 721  ┆ 2023-01-10     ┆ 52       ┆ OW        ┆ … ┆ null    ┆ false          ┆ false ┆ 23899      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 462  ┆ 2023-06-11     ┆ 37       ┆ GL        ┆ … ┆ unknown ┆ true           ┆ false ┆ 25287      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 1295 ┆ 2023-02-21     ┆ 4        ┆ AR        ┆ … ┆ BQ.1    ┆ false          ┆ false ┆ 8089       │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
│ 1499 ┆ 2023-04-05     ┆ 48       ┆ UR        ┆ … ┆ null    ┆ true           ┆ true  ┆ 24565      │
│      ┆ 00:00:00       ┆          ┆           ┆   ┆         ┆                ┆       ┆            │
└──────┴────────────────┴──────────┴───────────┴───┴─────────┴────────────────┴───────┴────────────┘

In [ ]:
test = client.get_dummy_dataset(nb_rows=NB_ROWS, seed = SEED)
test.dtypes


id                          Int64
date               datetime64[ns]
temporal                    Int64
georegion          string[python]
agegroup           string[python]
sex                string[python]
testType           string[python]
testResult         string[python]
country            string[python]
subType            string[python]
hospitalization           boolean
death                     boolean
patient_id                  Int64
dtype: object

In [ ]:
context = client.get_context(epsilon=1.0)

In [ ]:
context


Context(
    accountant = Measurement(
        input_domain   = FrameDomain(id: i64, date: datetime[ns], temporal: i64, georegion: str, agegroup: str, sex: str, testType: str, testResult: str, country: str, subType: str, hospitalization: bool, death: bool, patient_id: i64; margins=[{}, {col("id")}, {col("date")}, {col("temporal")}, {col("georegion")}, {col("agegroup")}, {col("sex")}, {col("testType")}, {col("testResult")}, {col("country")}, {col("subType")}, {col("hospitalization")}, {col("death")}, {col("patient_id")}, {col("sex"), col("agegroup")}, {col("sex"), col("georegion")}, {col("subType"), col("sex")}, {col("agegroup"), col("georegion")}]),
        input_metric   = SymmetricDistance(),
        output_measure = MaxDivergence),
    d_in       = 52,
    d_mids     = [1.0],
    d_out      = None)

## Step 4: Prepare the pipeline

It is necessary to prepare the pipeline before sending the query to the client.

In [ ]:
import polars as pl
pl.__name__, pl.__version__

('polars', '1.32.0')

* basic computations (mean, sum, etc.)

* basic on dates

* group by / agg

* with_columns (row-wise)

* join

* drop

* filter / select

* sort


### basic computation

a. Dataframe length

In [ ]:
plan = context.query().select(dp.len())

In [ ]:
plan.release().collect()

len
u32
100


b. sum

In [ ]:
context = client.get_context(epsilon=1.0)

In [ ]:
temporal_min, temporal_max = client.get_column_bounds("temporal")
temporal_min, temporal_max

(1, 52)

In [ ]:
plan = context.query().select(pl.col("temporal").dp.sum(bounds=(temporal_min, temporal_max)))

In [ ]:
# result on dummy dataset with dp
plan.release().collect()

temporal
i64
3866


In [ ]:
# result on dummy dataset without dp
dummy_lf.select(pl.col("temporal").sum()).collect()

temporal
i64
5392


In [ ]:
# actual result on remote server (with DP)
res = client.opendp.query(plan, epsilon=1.0)
print(res.result.value)

shape: (1, 1)
┌──────────┐
│ temporal │
│ ---      │
│ i64      │
╞══════════╡
│ 1327366  │
└──────────┘

### Aggregation

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)

In [ ]:
# Count the number of death per sex and age_group
plan = (
    context.query()
    .group_by(["sex", "agegroup"])
    .agg([
        pl.col("death").cast(int).dp.sum(bounds=(0,1))
    ])
)
result = plan.release().collect()
result.sort("death", descending=True)

sex,agegroup,death
str,str,i64
"""unknown""","""other""",222
"""female""","""80+""",163
"""other""","""30 - 39""",86
"""other""","""0 - 9""",80
"""other""","""60 - 69""",68
…,…,…
"""female""","""other""",-81
"""female""","""40 - 49""",-81
"""male""","""10 - 19""",-97


### `with_columns`

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)

In [ ]:
breaks = [13, 26, 39]
labels = pl.Series("quarter", list(range(len(breaks) + 1)), dtype=pl.UInt32)
plan = (
    context.query()
    .with_columns(
        pl.col.temporal.cut(
            breaks=breaks, left_closed=True
        ).to_physical().alias("quarter")
    )
    .group_by(pl.col.quarter)
    .agg([
        pl.col("death").cast(int).dp.sum(bounds=(0,1))
    ])
    .with_keys(pl.LazyFrame([labels]))
)
release = plan.release()

In [ ]:
release.collect().sort("quarter")

quarter,death
u32,i64
0,37
1,-66
2,-33
3,113


In [ ]:
res = client.opendp.query(plan, epsilon = 1.0, delta = 1e-6)
res.result.value.sort("quarter")

quarter,death
i64,i64
0,-99
1,-68
2,-117
3,77


### Datetime

a. Group by `year`

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)

In [ ]:
# Group by year and count the number of rows per year
plan = (
    context.query()
    .with_columns(YEAR=pl.col.date.dt.year(), MONTH=pl.col.date.dt.month())
    .group_by("YEAR")
    .agg(dp.len())
)

In [ ]:
# Check plan works on local dummy dataset
plan.release().collect()

YEAR,len
i32,u32


In [ ]:
# apply plan on remote server (private data)
res = client.opendp.query(plan, epsilon=1.0, delta=1e-06)
print(res.result.value)

shape: (2, 2)
┌──────┬───────┐
│ YEAR ┆ len   │
│ ---  ┆ ---   │
│ i64  ┆ i64   │
╞══════╪═══════╡
│ 2022 ┆ 24864 │
│ 2023 ┆ 25200 │
└──────┴───────┘

b. Group by "month"

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)
plan = (
    context.query()
    .with_columns(YEAR=pl.col.date.dt.year(), MONTH=pl.col.date.dt.month())
    .group_by("MONTH")
    .agg(dp.len())
)

# apply plan on remote server (private data)
res = client.opendp.query(plan, epsilon=1.0, delta=1e-06)
print(res.result.value.sort("MONTH"))

shape: (9, 2)
┌───────┬──────┐
│ MONTH ┆ len  │
│ ---   ┆ ---  │
│ i64   ┆ i64  │
╞═══════╪══════╡
│ 1     ┆ 8428 │
│ 2     ┆ 6876 │
│ 3     ┆ 6246 │
│ 4     ┆ 3062 │
│ 8     ┆ 1403 │
│ 9     ┆ 2415 │
│ 10    ┆ 5490 │
│ 11    ┆ 7346 │
│ 12    ┆ 8409 │
└───────┴──────┘

### `filter`

a. basic filter

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)

In [ ]:
plan = (
    context.query()
    .filter(pl.col.sex == "female")
    .filter(pl.col("temporal") >= 26)
    .select(
        pl.col.death.cast(int)
        .dp.sum(bounds=(0,1))
    )
)
release = plan.release()

In [ ]:
# dummy result with dp
release.collect()

death
i64
14


In [ ]:
# dummy result without dp

(dummy_lf
    .filter(pl.col.sex == "female")
    .filter(pl.col("temporal") >= 26)
    .select(
        pl.col.death
        .sum()
    )
).collect()

death
u32
10


In [ ]:
res = client.opendp.query(plan, epsilon=1.0, delta=1e-6)
res.result.value

death
i64
-60


2. `n_unique`

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)

In [ ]:
plan = context.query().select(pl.col.id.dp.n_unique())
plan.release().collect().item() 

196

### `join`

In [ ]:
context = client.get_context(epsilon=1.0, delta=1e-06)

In [ ]:
breaks = [13, 26, 39]
labels = pl.LazyFrame(pl.Series("quarter", list(range(len(breaks) + 1)), dtype=pl.UInt32))

In [ ]:
plan = (
    context.query()
    .with_columns(
        pl.col.temporal.cut(
            breaks=breaks, left_closed=True
        ).to_physical().alias("quarter")
    )
    .group_by(pl.col.quarter)
    .agg([
        pl.col("death").cast(int).dp.sum(bounds=(0,1))
    ])
    .join(labels, how="right", on="quarter")
)
collect = plan.release().collect()

In [ ]:
collect.sort("quarter")

death,quarter
i64,u32
-14,0
28,1
74,2
-144,3


In [ ]:
res = client.opendp.query(plan, epsilon=1.0, delta=1e-6)
res.result.value.sort("quarter")

death,quarter
i64,i64
103,0
85,1
101,2
6,3
